In [3]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = (
    project_root
    / "data"
    / "iavd_1000_incidents_raw_target.parquet"
)

df = pd.read_parquet(data_path)

print("Размерность:", df.shape)
print("\nТипы данных:")
print(df.dtypes.value_counts())

Размерность: (1000, 97)

Типы данных:
object                 77
datetime64[us, UTC]    11
float64                 5
int64                   4
Name: count, dtype: int64


In [4]:
# Названия всех столбцов
print("Столбцы датасета:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2}. {col}")

# Пропущенные значения
missing = df.isna().mean().mul(100).sort_values(
    ascending=False
)

print("\nТоп-10 столбцов по доле пропусков:")
print(missing.head(10))

# Проверка идентификаторов
print("\nДубликаты EntityId:")
print(df["EntityId"].duplicated().sum())

Столбцы датасета:
 1. EntityId
 2. Создан
 3. Дата
 4. Вердикт
 5. Дата закрытия (факт)
 6. Попытка эксплуатации
 7. Тип инцидента
 8. Список вредоносных команд
 9. Критичность
10. Ответственный
11. Пиктограмма IRP
12. Прошлый ответственный
13. Типы угроз
14. Группа устранения
15. Рекомендации по плейбуку
16. Фаза реагирования
17. Теги
18. Источник инцидента
19. Вендор из сием
20. Организация
21. Наименование
22. suser
23. filePath
24. processParentPath
25. objectLinks
26. Ответсвенный по РП
27. Принятые меры
28. objectValue
29. Значение Атаки (группировка)
30. Решение
31. Алерт
32. Правило корреляции SIEM
33. Критичность инцидента в источнике
34. Полное имя
35. Статус инцидента из SIEM
36. Значение Инцидента (группировка)
37. Значение Инцидента (группировка по типу)
38. Файловый ресурс
39. Имя файла-IRP
40. Принятые меры из SIEM
41. Команда
42. CMDLINE
43. Конвейер SIEM
44. Описание инцидента
45. Hostname Цели
46. categoryHigh
47. categoryLow
48. categoryMain
49. commandName
50. duser

In [5]:
missing = df.isna().sum().to_frame(name="Пропусков")

missing["Доля, %"] = (
    missing["Пропусков"] / len(df) * 100
).round(2)

missing = missing.sort_values(
    by="Доля, %", ascending=False
)

display(missing.head(15))

,Пропусков,"Доля, %"
Прошлый ответственный,1000,100.0
Эскалировался на L2?,1000,100.0
Ошибки,1000,100.0
filePath,999,99.9
Ссылка в NAD (actual),999,99.9
Привилегии аккаунта,999,99.9
Вендор из сием,998,99.8
Список вредоносных команд,998,99.8
commandName,998,99.8
Пауза,996,99.6


In [7]:
# Загружаем полный датасет
full_path = (
    project_root
    / "data"
    / "iavd_incidents_raw_target.parquet"
)

df_full = pd.read_parquet(full_path)

print("Размерность:", df_full.shape)
print("Дубликаты EntityId:",
      df_full["EntityId"].duplicated().sum())

# Проверяем пропуски
missing_full = pd.DataFrame({
    "Пропусков": df_full.isna().sum(),
    "Доля, %": (
        df_full.isna().mean() * 100
    ).round(2)
})

display(
    missing_full.sort_values(
        by="Доля, %", ascending=False
    ).head(15)
)

Размерность: (240295, 97)
Дубликаты EntityId: 0


,Пропусков,"Доля, %"
Ошибки,240282,99.99
Типы угроз,239642,99.73
Прошлый ответственный,239200,99.54
Успех brute force,238885,99.41
Эскалировался на L2?,238539,99.27
Список вредоносных команд,236437,98.39
filePath,234371,97.53
False Positive,230639,95.98
Привилегии аккаунта,228114,94.93
Полное имя,221004,91.97


In [8]:
columns = [
    "Вердикт",
    "Критичность",
    "Тип инцидента",
    "Источник инцидента",
    "Фаза реагирования"
]

for col in columns:
    print(f"\n{'=' * 50}")
    print(f"Столбец: {col}")
    print(f"Пропусков: {df_full[col].isna().sum()}")
    print(f"Уникальных значений: {df_full[col].nunique()}")

    display(
        df_full[col].value_counts(dropna=False).head(10)
    )


Столбец: Вердикт
Пропусков: 0
Уникальных значений: 7


Вердикт
False Positive                       150675
Не указан                             54870
Решён                                 28606
Дубликат                               5819
Не решён                                316
Закрыт по совокупности инцидентов         8
Закрыт из родительской атаки              1
Name: count, dtype: int64


Столбец: Критичность
Пропусков: 0
Уникальных значений: 5


Критичность
Средняя        158159
Высокая         45510
Низкая          32497
Критическая      4079
Отсутствует        50
Name: count, dtype: int64


Столбец: Тип инцидента
Пропусков: 116
Уникальных значений: 99


Тип инцидента
Выполнение подозрительной команды                            45521
Общий Инцидент ИБ                                            40579
Сканирование хоста                                           23751
Подозрительный трафик                                        16939
Проведение разведки                                          12762
Сканирование сети                                             8530
Вирус не вылечен                                              8194
Подозрительный процесс                                        6965
Изменение расписаний (создание/удаление/изменение задачи)     5711
Ошибки в работе ПО                                            5395
Name: count, dtype: int64


Столбец: Источник инцидента
Пропусков: 5
Уникальных значений: 5


Источник инцидента
MaxPatrol SIEM                       142549
Ankey SIEM NG                         97551
Ручной ввод                             126
Почта                                    35
Ankey SIEM NG~~||~~MaxPatrol SIEM        29
None                                      5
Name: count, dtype: int64


Столбец: Фаза реагирования
Пропусков: 0
Уникальных значений: 9


Фаза реагирования
Закрыт                         239557
Закрыт~~||~~Проанализирован       519
Ожидание                          103
Закрыт~~||~~Сдерживание            55
В работе                           53
Классификация~~||~~Ожидание         4
Новый                               2
Устранение                          1
В работе~~||~~Классификация         1
Name: count, dtype: int64

In [9]:
created = pd.to_datetime(
    df_full["Создан"], errors="coerce", utc=True
)

closed = pd.to_datetime(
    df_full["Дата закрытия (факт)"],
    errors="coerce",
    utc=True
)

print("Период данных:")
print("Начало:", created.min())
print("Конец:", created.max())

print("\nПропуски:")
print("Создан:", created.isna().sum())
print("Дата закрытия:", closed.isna().sum())

# Длительность между созданием и закрытием
duration_hours = (
    closed - created
).dt.total_seconds() / 3600

print("\nПроверка длительности:")
print("Отрицательных:", (duration_hours < 0).sum())
print("Нулевых:", (duration_hours == 0).sum())
print("Медиана, часов:", duration_hours.median())
print("Максимум, часов:", duration_hours.max())

Период данных:
Начало: 2024-04-12 17:06:04.854999+00:00
Конец: 2025-09-03 12:21:17.348999+00:00

Пропуски:
Создан: 0
Дата закрытия: 42139

Проверка длительности:
Отрицательных: 12331
Нулевых: 0
Медиана, часов: 0.17796791652777777
Максимум, часов: 6353.965331666666


In [10]:
# Только отрицательные длительности
negative = duration_hours[duration_hours < 0]

print("Количество:", len(negative))

print("\nСтатистика отрицательных длительностей (ч):")
print(negative.describe())

print("\nСамые частые значения (округление до часа):")
print(negative.round().value_counts().head(10))

print("\nПримеры карточек:")
display(
    df_full.loc[
        duration_hours < 0,
        ["EntityId", "Создан",
         "Дата закрытия (факт)", "Фаза реагирования"]
    ].head(5)
)

Количество: 12331

Статистика отрицательных длительностей (ч):
count    12331.000000
mean        -0.025848
std          0.015555
min         -0.072271
25%         -0.037505
50%         -0.024409
75%         -0.012933
max         -0.000005
dtype: float64

Самые частые значения (округление до часа):
-0.0    12331
Name: count, dtype: int64

Примеры карточек:


,EntityId,Создан,Дата закрытия (факт),Фаза реагирования
8207,486908,2025-05-05 12:05:38.255000+00:00,2025-05-05 12:02:53.290000+00:00,Закрыт
45274,293830,2024-07-01 15:33:20.802000+00:00,2024-07-01 15:33:13.796000+00:00,Закрыт
45287,293853,2024-07-01 16:03:25.900000+00:00,2024-07-01 16:03:20.014000+00:00,Закрыт
45377,293971,2024-07-01 17:53:27.739000+00:00,2024-07-01 17:53:22.156000+00:00,Закрыт
45387,293818,2024-07-01 15:23:25.580000+00:00,2024-07-01 15:23:19.476000+00:00,Закрыт


In [11]:
# Создаём копию исходного DataFrame
df_prepared = df_full.copy()

# Рассчитываем длительность между созданием и закрытием
duration = (
    df_prepared["Дата закрытия (факт)"]
    - df_prepared["Создан"]
)

# Отмечаем карточки с отрицательной длительностью
df_prepared["invalid_duration"] = duration < pd.Timedelta(0)

# Длительность в часах.
# Отрицательные значения и отсутствующие даты превращаются в NaN.
df_prepared["duration_hours"] = (
    duration.where(duration >= pd.Timedelta(0))
    .dt.total_seconds() / 3600
)

print("Карточек с отрицательной длительностью:",
      df_prepared["invalid_duration"].sum())

print("Карточек с корректной длительностью:",
      df_prepared["duration_hours"].notna().sum())

print("Медиана корректной длительности, ч:",
      df_prepared["duration_hours"].median())

Карточек с отрицательной длительностью: 12331
Карточек с корректной длительностью: 185825
Медиана корректной длительности, ч: 0.20535666666666666


In [12]:
time_columns = ["Время решения", "Время Решения"]

for col in time_columns:
    if col not in df_full.columns:
        print(f"Столбец {col} отсутствует")
        continue

    t = pd.to_numeric(df_full[col], errors="coerce")

    print(f"\n=== {col} ===")
    print("Пропусков:", t.isna().sum())
    print("Отрицательных:", (t < 0).sum())
    print("Нулевых:", (t == 0).sum())

    print("\nСтатистика в исходных единицах:")
    print(t.describe(percentiles=[0.5, 0.9, 0.99]))

    print("\nМедиана, если значения в секундах (ч):",
          t.median() / 3600)


=== Время решения ===
Пропусков: 1672
Отрицательных: 20
Нулевых: 33457

Статистика в исходных единицах:
count    2.386230e+05
mean     1.433056e+11
std      4.757853e+11
min     -1.082165e+07
50%      1.285568e+05
90%      3.189109e+07
99%      1.726209e+12
max      1.755077e+12
Name: Время решения, dtype: float64

Медиана, если значения в секундах (ч): 35.710210249999996

=== Время Решения ===
Пропусков: 0
Отрицательных: 44344
Нулевых: 1

Статистика в исходных единицах:
count    2.402950e+05
mean    -1.700667e+18
std      3.578645e+18
min     -9.223372e+18
50%      1.727125e+15
90%      1.748380e+15
99%      1.756329e+15
max      1.756918e+15
Name: Время Решения, dtype: float64

Медиана, если значения в секундах (ч): 479757033611.1111
